In [3]:
import pandas as pd
import numpy as np
import json
import re, pytz, os, requests, sys
from pathlib import Path
from datetime import datetime
import sys
sys.path.append("/workspaces/service-data")

from src.clean import clean_percentage, clean_fiscal_yr, normalize_string, standardize_column_names
from src.load import load_csv
from src.export import export_to_csv
from src.merge import merge_si, merge_ss
from src.utils import dept_list, program_list
from main import get_config

import pandas as pd
import numpy as np
import pytz
from pathlib import Path

base_dir = Path.cwd()
parent_dir = base_dir.parent

In [4]:
config=get_config()
si_path = parent_dir / 'outputs' / 'si.csv'
si = pd.read_csv(si_path, keep_default_na=False, na_values='', delimiter=';', engine='python', skipfooter=2)

si_qa_path = parent_dir / 'outputs' / 'qa' / 'si_qa.csv'
si_qa_report = pd.read_csv(si_qa_path, keep_default_na=False, na_values='', delimiter=';', engine='python', skipfooter=2)

sid_registry_path = parent_dir / 'inputs' / 'sid_registry.csv'
sid_registry = load_csv(sid_registry_path, config)

In [5]:
# === QA check: unregistered service ID
# This service id is not registered in the service id registry
si['qa_unregistered_sid'] = ~si['service_id'].isin(sid_registry['service_id'])

# === QA check: reused service ID
# This service id is being reported by the wrong organization
# Bring in the service id, organization, and transferred date, if applicable, from sid registry
si = pd.merge(si, sid_registry[['service_id', 'org_id', 'date_transferred']], on=['service_id', 'org_id'], how='left', indicator=True)

si['fiscal_yr_end_date'] = pd.to_datetime(si['fiscal_yr'].str.split('-').str[1]+'-03-31')
si['date_transferred'] = pd.to_datetime(si['date_transferred'])

# Consider the following 4 predicates:
# A = registry contains service ID (t/f)
# B = registry contains service-org combination (t/f)
# C = org-service combo in inventory has transferred date in registry (t/f) 
# D = org-service combo is reported in fiscal year that ends before the transfer date (t/f)
# qa_reused_id = A & ( !B | ( C & !D )) -> determined based on truth matrix and expected values

# # Verification using predicates
# si['qa_pA'] = si['service_id'].isin(sid_registry['service_id'])
# si['qa_pB'] = si['_merge'] == 'both'
# si['qa_pC'] = ~(si['date_transferred'].isnull())
# si['qa_pD'] = (si['fiscal_yr_end_date'] < si['date_transferred'])

# si['qa_reused_id'] = si['qa_pA'] & (
#     ~si['qa_pB'] | (
#         si['qa_pC'] & 
#         ~si['qa_pD']
#     )
# )

# Without intermediary predicates:
si['qa_reused_id'] = si['service_id'].isin(sid_registry['service_id']) & (
    ~(si['_merge'] == 'both') | (
        (~(si['date_transferred'].isnull())) & 
        ~((si['fiscal_yr_end_date'] < si['date_transferred']))
    )
)

si.drop(columns=['_merge'], inplace=True)